# DDPM Reproduction & Inference Demo
**Course:** CS-4112 Deep Learning — FAST-NUCES Islamabad  
**Project:** Denoising Diffusion Probabilistic Models (DDPM) on CIFAR-10

This notebook serves as a chronological proof of reproduction. It is divided into two parts:
1. **Scratch Reproduction:** Proof of training from scratch for 20,000 steps using the official paper's architecture.
2. **Research-Quality Inference:** Using official pretrained weights to validate high-fidelity generation and DDIM acceleration.

### 1. Setup & Installation

In [ ]:
!pip install -q diffusers accelerate torch torchvision matplotlib tensorboardX tqdm

### 2. Imports & Path Configuration

In [ ]:
import torch
import os
import sys
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from diffusers import DDPMPipeline, DDIMScheduler

# Add the reproduction folder to path to import original architecture
repro_path = os.path.abspath("repo/original_architecture_reproduction")
if repro_path not in sys.path:
    sys.path.append(repro_path)

from model import UNet
from diffusion import GaussianDiffusionSampler

--- 
## Part 1: Official Repo Reproduction (Our 20,000 Step Weights)

To satisfy the instructor's requirement of using the official paper's code, we trained the original U-Net from scratch. Below, we load our checkpoint from **Step 20,000** (approx. 2.5% of full training).

**Technical Note:** The full 548MB checkpoint is stored locally in the `logs/` directory but is excluded from GitHub due to size constraints. If running this notebook locally, it will perform live generation. Otherwise, it displays the pre-generated visual proof below.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_path = "repo/original_architecture_reproduction/logs/DDPM_Reproduction_Attempt/ckpt.pt"

if os.path.exists(ckpt_path):
    print(f"Loading our 20k-step checkpoint from {ckpt_path}...")
    model = UNet(ch=128, ch_mult=[1, 2, 2, 2], attn=[1], num_res_blocks=2, dropout=0.1).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt)
    
    sampler = GaussianDiffusionSampler(model, beta_1=1e-4, beta_T=0.02, T=1000, img_size=32, 
                                      mean_type='epsilon', var_type='fixedlarge').to(device)
    
    print("Generating samples from scratch weights...")
    model.eval()
    with torch.no_grad():
        x_T = torch.randn((8, 3, 32, 32)).to(device)
        samples = sampler(x_T)
        samples = (samples + 1) / 2
    
    grid = make_grid(samples, nrow=8).cpu().permute(1, 2, 0).numpy()
    plt.figure(figsize=(15, 3))
    plt.imshow(grid)
    plt.title("Live Generated: Our Scratch Reproduction (Step 20,000)")
    plt.axis("off")
    plt.show()
else:
    print("Note: Local checkpoint (548MB) is stored on the local machine.")
    print("Displaying the pre-generated visual proof from the training logs below:")
    
    sample_img_path = "repo/original_architecture_reproduction/logs/DDPM_Reproduction_Attempt/sample/20000.png"
    if os.path.exists(sample_img_path):
        from PIL import Image
        img = Image.open(sample_img_path)
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title("Visual Proof: Sample at Step 20,000 (from logs)")
        plt.axis("off")
        plt.show()


--- 
## Part 2: Research-Quality Results (Full Pretrained Scale)

After verifying the pipeline with the 20k-step run, we pivot to the full pretrained checkpoint to conduct our Assignment 3 experiments. This represents the *exact same model* after the full 800,000 training steps.

In [ ]:
model_id = "google/ddpm-cifar10-32"
print(f"Loading full pretrained model to {device}...")
ddpm = DDPMPipeline.from_pretrained(model_id).to(device)

num_images = 8
with torch.no_grad():
    images = ddpm(batch_size=num_images).images

# Visualization
fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
for i, img in enumerate(images):
    axes[i].imshow(img)
    axes[i].axis("off")
plt.suptitle("Full Training Results (800,000 Steps) - Research Baseline")
plt.show()

### Accelerated Sampling (DDIM)
Demonstrating the speedup from our Assignment 3 findings (50 steps instead of 1000).

In [ ]:
ddpm.scheduler = DDIMScheduler.from_config(ddpm.scheduler.config)
with torch.no_grad():
    images_ddim = ddpm(batch_size=num_images, num_inference_steps=50, eta=0.0).images

fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
for i, img in enumerate(images_ddim):
    axes[i].imshow(img)
    axes[i].axis("off")
plt.suptitle("DDIM Accelerated Inference (50 Steps)")
plt.show()